In [ ]:
#!pip install --upgrade git+https://github.com/ColibrITD-SAS/landscape_tools@dev

In [ ]:
!pip install landscape_tools

# Imports

In [ ]:
from landscape_tools import landscape_visualization as lv
import numpy as np
import matplotlib.pyplot as plt
from qiskit import QuantumCircuit
from qiskit.circuit import ParameterVector
from qiskit.quantum_info import Statevector
from scipy.optimize import minimize

# Example of a problem

## Probability distribution

In [ ]:
def get_prob_distribution(params, ansatz, theta_params):

    param_dict = dict(zip(theta_params, params))                               
    # param_dict = {
    #     θ_0: 0.5,
    #     θ_1: 1.2
    # }

    bound_circuit = ansatz.assign_parameters(param_dict, inplace=False)

    state = Statevector.from_instruction(bound_circuit)
    probs = state.probabilities()

    return np.asarray(probs, dtype=float)

In [ ]:
n_qubits = 5
dim = 2 ** n_qubits

p_target = np.zeros(dim)

# index 0 -> |000>, index 1 -> |001>, ..., index 7 -> |111>
p_target[0] = 0.40  # |000>
p_target[3] = 0.25  # |011>
p_target[7] = 0.35  # |111>

p_target = p_target / p_target.sum()    # normalize to ensure probabilities sum to 1

plt.bar(range(dim), p_target)
plt.xlabel("state |x>")
plt.ylabel("target probability")
plt.title("Target distribution")
plt.show()

## Quantum circuit

In [ ]:
def build_hea_ansatz(n_qubits, n_layers):
    """
    Hardware Efficient Ansatz :
    - RY + RZ on each qubit
    - CNOT chain
    """
    qc = QuantumCircuit(n_qubits)

    n_params = 2 * n_qubits * n_layers
    theta = ParameterVector("θ", n_params)

    k = 0
    for _ in range(n_layers):
        for q in range(n_qubits):
            qc.ry(theta[k], q)
            k += 1
            qc.rz(theta[k], q)
            k += 1

        for q in range(n_qubits - 1):
            qc.cx(q, q + 1)

    return qc, theta

In [ ]:
n_layers = 2

ansatz, theta_params = build_hea_ansatz(n_qubits, n_layers)

n_params = len(theta_params)

print("Number of parameters:", n_params)
ansatz.draw("mpl")

In [ ]:
params_test = np.random.uniform(0, 2*np.pi, size=n_params)

p_test = get_prob_distribution(params_test, ansatz, theta_params)

print(p_test)
print("Sum =", p_test.sum())

## Loss function

In [ ]:
def mse_loss(params, ansatz, theta_params, p_target):
    p_model = get_prob_distribution(params, ansatz, theta_params)
    return np.sum((p_model - p_target) ** 2)

## Optimization

In [ ]:
theta_history = []
loss_history = []

def objective(params):
    loss = mse_loss(params, ansatz, theta_params, p_target)
    return loss

def callback(params):
    theta_history.append(np.copy(params))
    loss_history.append(objective(params))

rng = np.random.default_rng(seed=42)

initial_params = rng.uniform(
    low=0.0,
    high=2*np.pi,
    size=n_params
)

In [ ]:
result = minimize(
    objective,
    initial_params,
    method="COBYLA",
    callback=callback,
    options={
        "maxiter": 3000,
        "rhobeg": 0.5,
        "tol": 1e-6,
        "disp": True
    }
)

In [ ]:
theta_history = np.array(theta_history)
loss_history = np.array(loss_history)

print("theta_history shape:", theta_history.shape)
print("loss_history shape:", loss_history.shape)
plt.plot(loss_history)
plt.xlabel("iter")
plt.ylabel("MSE Loss")
plt.title("Loss evolution")
plt.grid(True)
plt.show()

In [ ]:
p_final = get_prob_distribution(result.x, ansatz, theta_params)

x = np.arange(dim)

plt.figure(figsize=(8, 4))
plt.bar(x - 0.2, p_target, width=0.4, label="Target")
plt.bar(x + 0.2, p_final, width=0.4, label="Model")
plt.xlabel("state |x>")
plt.ylabel("probability")
plt.title("Target vs learnt distribution")
plt.legend()
plt.show()

# Landscape visualization

In [ ]:
def make_mse_objective(ansatz, theta_params, p_target):
    
    def objective(params):
        return mse_loss(params, ansatz, theta_params, p_target)
    
    return objective

objective = make_mse_objective(ansatz, theta_params, p_target)

In [ ]:
d1, d2 = lv.random_mixed_directions(len(theta_history[-1]))
paramsf = theta_history[-1]

In [ ]:
lv.loss_scan_1d(
    params=paramsf,
    direction=d1,
    loss_function=objective,
    n_steps=3000,
    end_points=(-0.01, 0.01),
    n_jobs=-1
)

In [ ]:
lv.loss_scan_2d_3d(
    params=paramsf,
    direction1=d1,
    direction2=d2,
    loss_function=objective,
    n_steps=300,
    end_points_x=(-0.01, 0.01),
    end_points_y=(-0.01, 0.01),
    plot3D=True,
    n_jobs=-1
)

In [ ]:
res = lv.perform_pca_and_analysis(
    theta_history,
    objective,
    n_steps=300,
    offset=0.5,
    n_top=12,
    circuit=ansatz,
)